In [ ]:
Exploring the data from Jobicy for Market skill demand

In [1]:
import requests
import pandas as pd
import json
import time

from pathlib import Path
from datetime import datetime, timezone

# Jobicy API
url = "https://jobicy.com/api/jobs"

job_roles = [
    "Data Analyst",
    "Data Scientist",
    "Data Engineer",
    "Software Engineer",
    "AI Engineer"
]

all_jobs = []

# Create folder to store downloaded data
output_folder = Path("../data/jobicy")
output_folder.mkdir(parents=True, exist_ok=True)

for role in job_roles:

    print("\nCollecting:", role)

    params = {
        "geo": "usa",
        "tag": role,
        "count": 100
    }

    try:
        response = requests.get(
            url,
            params=params,
            timeout=30
        )

        response.raise_for_status()

        data = response.json()

        if data.get("success") is False:
            print("API Error:", data.get("error"))
            continue

        jobs = data.get("jobs", [])

        print("Jobs found:", len(jobs))

        # Save original API response
        filename = role.lower().replace(" ", "_")

        with open(
            output_folder / f"{filename}.json",
            "w",
            encoding="utf-8"
        ) as file:
            json.dump(data, file, indent=2)

        # Combine job records
        for job in jobs:

            record = job.copy()

            record["searched_role"] = role
            record["collected_at"] = (
                datetime.now(timezone.utc).isoformat()
            )

            all_jobs.append(record)

    except requests.RequestException as e:
        print("Request failed:", e)

    time.sleep(1)

# Convert collected jobs into DataFrame
df = pd.DataFrame(all_jobs)

if not df.empty:

    # Remove repeated Jobicy job IDs
    df = df.drop_duplicates(subset=["id"])

    # Save combined data
    df.to_csv(
        output_folder / "jobicy_all_jobs.csv",
        index=False
    )

    print("\nTotal unique jobs:", len(df))

    display(df.head())

else:
    print("No jobs collected.")


Collecting: Data Analyst
Jobs found: 5

Collecting: Data Scientist
Jobs found: 8

Collecting: Data Engineer
Jobs found: 13

Collecting: Software Engineer
Jobs found: 44

Collecting: AI Engineer
Jobs found: 15

Total unique jobs: 74


,id,url,jobSlug,jobTitle,companyName,companyLogo,jobIndustry,jobType,jobGeo,jobLevel,jobExcerpt,jobDescription,pubDate,salaryMin,salaryMax,salaryCurrency,salaryPeriod,searched_role,collected_at
0,147852,https://jobicy.com/jobs/147852-principal-analy...,147852-principal-analyst-data-integration,"Principal Analyst, Data Integration",H1,https://jobicy.com/data/server-nyc0409/galaxy/...,[Data Science & Analytics],[Full-Time],USA,Senior,"At H1, we believe access to the best healthcar...","<div>At <a href=""https://jobicy.com/company/h1...",2026-10-31T03:30:19+00:00,145000.0,170000.0,USD,yearly,Data Analyst,2026-10-31T18:52:52.298962+00:00
1,158025,https://jobicy.com/jobs/158025-senior-data-pro...,158025-senior-data-product-analyst,Senior Data Product Analyst,YipitData,https://jobicy.com/data/server-nyc0409/galaxy/...,[Data Science & Analytics],[Full-Time],USA,Senior,"Product · New York, NY (Remote-Friendly) About...","<p><em>Product · New York, NY (Remote-Friendly...",2026-10-30T16:26:51+00:00,NaN,NaN,NaN,NaN,Data Analyst,2026-10-31T18:52:52.298989+00:00
2,157762,https://jobicy.com/jobs/157762-senior-data-ana...,157762-senior-data-analyst-medtech,Senior Data Analyst (MedTech),YipitData,https://jobicy.com/data/server-nyc0409/galaxy/...,[Data Science & Analytics],[Full-Time],USA,Senior,About Us: YipitData is the leading market rese...,\n<p><strong>About Us:</strong></p>\n<p><a hre...,2026-10-29T05:48:09+00:00,175000.0,200000.0,USD,yearly,Data Analyst,2026-10-31T18:52:52.298996+00:00
3,157759,https://jobicy.com/jobs/157759-senior-data-ana...,157759-senior-data-analyst-3,Senior Data Analyst,YipitData,https://jobicy.com/data/server-nyc0409/galaxy/...,[Data Science & Analytics],[Full-Time],USA,Senior,About Us: YipitData is the leading market rese...,\n<p><strong>About Us:</strong></p>\n<p><a hre...,2026-10-29T05:47:48+00:00,170000.0,190000.0,USD,yearly,Data Analyst,2026-10-31T18:52:52.299001+00:00
4,156941,https://jobicy.com/jobs/156941-senior-data-ana...,156941-senior-data-analyst-marketing,Senior Data Analyst - Marketing,Supabase,https://jobicy.com/data/server-nyc0409/galaxy/...,[Data Science & Analytics],[Full-Time],Anywhere,Senior,About Supabase Supabase is the Postgres develo...,"<h3><strong>About <a href=""https://jobicy.com/...",2026-10-25T10:45:07+00:00,NaN,NaN,NaN,NaN,Data Analyst,2026-10-31T18:52:52.299006+00:00


Step 1: Clean Jobicy Live API Data

In [1]:
%pip install beautifulsoup4 requests

Note: you may need to restart the kernel to use updated packages.


In [2]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

# Jobicy API
url = "https://jobicy.com/api/jobs"

job_roles = [
    "Data Analyst",
    "Data Scientist",
    "Data Engineer",
    "Software Engineer",
    "AI Engineer"
]

all_jobs = []

# Collect postings
for role in job_roles:

    params = {
        "geo": "usa",
        "tag": role,
        "count": 100
    }

    try:
        response = requests.get(
            url,
            params=params,
            timeout=30
        )

        response.raise_for_status()

        data = response.json()

        jobs = data.get("jobs", [])

        print(role, ":", len(jobs), "jobs")

        for job in jobs:
            all_jobs.append(job.copy())

    except requests.RequestException as e:
        print("Error:", e)

# Convert JSON records to DataFrame
jobicy_clean = pd.DataFrame(all_jobs)

if jobicy_clean.empty:
    raise ValueError("No Jobicy records were collected.")

# Remove duplicate job IDs
jobicy_clean = jobicy_clean.drop_duplicates(
    subset=["id"]
).copy()

# Rename columns
jobicy_clean = jobicy_clean.rename(columns={
    "id": "job_id",
    "jobTitle": "job_title",
    "companyName": "company_name",
    "jobGeo": "location",
    "jobDescription": "description",
    "pubDate": "publication_date"
})

# Remove HTML tags
def clean_html(text):

    if pd.isna(text):
        return ""

    soup = BeautifulSoup(str(text), "html.parser")

    return " ".join(soup.stripped_strings)

jobicy_clean["description_clean"] = (
    jobicy_clean["description"].apply(clean_html)
)

# Standardize job titles
jobicy_clean["job_title"] = (
    jobicy_clean["job_title"]
    .astype("string")
    .str.strip()
)

# Standardize company names
jobicy_clean["company_name"] = (
    jobicy_clean["company_name"]
    .astype("string")
    .str.strip()
)

# Convert publication dates
jobicy_clean["publication_date"] = (
    pd.to_datetime(
        jobicy_clean["publication_date"],
        errors="coerce",
        utc=True
    )
)

# Display cleaned data
print("Total Unique Jobs:", len(jobicy_clean))

display(
    jobicy_clean[
        [
            "job_id",
            "job_title",
            "company_name",
            "location",
            "description_clean"
        ]
    ].head()
)

Data Analyst : 5 jobs
Data Scientist : 9 jobs
Data Engineer : 13 jobs
Software Engineer : 46 jobs
AI Engineer : 16 jobs
Total Unique Jobs: 79


,job_id,job_title,company_name,location,description_clean
0,147852,"Principal Analyst, Data Integration",H1,USA,"At H1 , we believe access to the best healthca..."
1,158025,Senior Data Product Analyst,YipitData,USA,"Product · New York, NY (Remote-Friendly) About..."
2,157762,Senior Data Analyst (MedTech),YipitData,USA,About Us: YipitData is the leading market rese...
3,157759,Senior Data Analyst,YipitData,USA,About Us: YipitData is the leading market rese...
4,156941,Senior Data Analyst - Marketing,Supabase,Anywhere,About Supabase Supabase is the Postgres develo...


Step 2: Extract technical skills from Jobicy

In [3]:
import re

# Initial skill dictionary
skill_patterns = {
    "Python": r"\bpython\b",
    "SQL": r"\bsql\b",
    "Tableau": r"\btableau\b",
    "Power BI": r"\bpower\s*bi\b",
    "Microsoft Excel": r"\b(?:microsoft excel|ms excel)\b",
    "AWS": r"\b(?:aws|amazon web services)\b",
    "PostgreSQL": r"\b(?:postgresql|postgres)\b",
    "Snowflake": r"\bsnowflake\b",
    "Docker": r"\bdocker\b",
    "Java": r"\bjava\b"
}

# Extract skills from job descriptions
def extract_skills(description):

    found_skills = []

    for skill, pattern in skill_patterns.items():

        if re.search(
            pattern,
            description,
            flags=re.IGNORECASE
        ):
            found_skills.append(skill)

    return found_skills


jobicy_clean["extracted_skills"] = (
    jobicy_clean["description_clean"].apply(
        extract_skills
    )
)

display(
    jobicy_clean[
        [
            "job_title",
            "extracted_skills"
        ]
    ].head(10)
)

,job_title,extracted_skills
0,"Principal Analyst, Data Integration","[SQL, AWS]"
1,Senior Data Product Analyst,[SQL]
2,Senior Data Analyst (MedTech),[]
3,Senior Data Analyst,"[Python, SQL]"
4,Senior Data Analyst - Marketing,"[Python, SQL, PostgreSQL]"
5,"Senior Data Scientist, MNO Data","[Python, SQL]"
6,"Data Scientist, Analytics (Technical Leadership)","[Python, SQL]"
7,Sr. Data Scientist,"[Python, SQL, AWS]"
8,"Senior Data Scientist, Product","[Python, SQL, AWS, PostgreSQL, Snowflake]"
9,"Data Scientist, Risk","[Python, SQL, AWS, Snowflake]"


Step 3: Prepare the Job-Skills table

In [4]:
# Create job-skill relationships
job_skills_clean = jobicy_clean[
    [
        "job_id",
        "extracted_skills"
    ]
].explode("extracted_skills")

# Rename column
job_skills_clean = job_skills_clean.rename(
    columns={
        "extracted_skills": "skill_name"
    }
)

# Remove rows without extracted skills
job_skills_clean = job_skills_clean.dropna(
    subset=["skill_name"]
)

# Remove duplicate job-skill pairs
job_skills_clean = job_skills_clean.drop_duplicates(
    subset=["job_id", "skill_name"]
)

print(
    "Total Job-Skill Relationships:",
    len(job_skills_clean)
)

display(job_skills_clean.head(20))

Total Job-Skill Relationships: 176


,job_id,skill_name
0,147852,SQL
0,147852,AWS
1,158025,SQL
3,157759,Python
3,157759,SQL
4,156941,Python
4,156941,SQL
4,156941,PostgreSQL
5,158241,Python
5,158241,SQL


In [ ]:
Step 4: Save your cleaned datasets

In [5]:
from pathlib import Path

# Output folder
output_folder = Path(
    r"C:\Users\belie\OneDrive\Desktop\ADT MySQL\Lab 7 Project Milestone\JobAvenir\data\cleaned"
)

output_folder.mkdir(
    parents=True,
    exist_ok=True
)
# Save Jobicy data
# Store only scalar columns to avoid CSV list serialization issues
jobicy_clean[
    [
        "job_id",
        "job_title",
        "company_name",
        "location",
        "description_clean",
        "publication_date"
    ]
].to_csv(
    output_folder / "jobicy_jobs_clean.csv",
    index=False
)

# Save extracted job-skill relationships
job_skills_clean.to_csv(
    output_folder / "job_skills_clean.csv",
    index=False
)

print("All cleaned datasets saved successfully!")

All cleaned datasets saved successfully!


In [ ]:
Automating the jobicy data process using Python's MySQL connector.

In [6]:
import mysql.connector

connection = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Sheetal@24",
    database="jobavenir_db"
)

cursor = connection.cursor()

cursor.execute("SELECT COUNT(*) FROM occupations")

result = cursor.fetchone()

print("Total occupations:", result[0])

cursor.close()
connection.close()

Total occupations: 0
